# Flow Size Distribution CDFs

This notebook plots the flow-count CDF and the byte-volume CDF for the Datamining, Hadoop, and Websearch flow size distributions.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import pandas as pd


DISTRIBUTIONS = {
    "Datamining": Path("DM.csv"),
    "Hadoop": Path("HD.csv"),
    "Websearch": Path("WS.csv"),
}


def load_distribution(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=["flow_size", "flow_cdf"])
    df = df.sort_values("flow_size", kind="stable").reset_index(drop=True)
    df["flow_pdf"] = df["flow_cdf"].diff().fillna(df["flow_cdf"])

    volume_pdf = df["flow_size"] * df["flow_pdf"]
    total_volume = volume_pdf.sum()
    if total_volume <= 0:
        raise ValueError(f"{path} has non-positive total volume")

    df["volume_cdf"] = volume_pdf.cumsum() / total_volume
    return df


distributions = {
    name: load_distribution(path)
    for name, path in DISTRIBUTIONS.items()
}


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
FONT_SIZE = 14

colors = {
    "Datamining": "tab:blue",
    "Hadoop": "tab:orange",
    "Websearch": "tab:green",
}


fig, ax = plt.subplots(figsize=(7, 3))
for name, df in distributions.items():
    ax.step(
        df["flow_size"],
        df["flow_cdf"],
        where="post",
        label=name,
        linewidth=2,
        color=colors[name],
    )

ax.set_xscale("log")
ax.set_xlabel("Flow size (bytes)", fontsize=FONT_SIZE)
ax.set_ylabel("CDF of number of flows", fontsize=FONT_SIZE)
ax.set_title("Flow-count CDF by workload", fontsize=FONT_SIZE)
ax.set_ylim(0, 1)
ax.tick_params(axis="both", labelsize=FONT_SIZE)
ax.legend(frameon=True, fontsize=FONT_SIZE)
fig.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(7, 3))
for name, df in distributions.items():
    ax.step(
        df["flow_size"],
        df["volume_cdf"],
        where="post",
        label=name,
        linewidth=2,
        color=colors[name],
    )

ax.set_xscale("log")
ax.set_xlabel("Flow size (bytes)", fontsize=FONT_SIZE)
ax.set_ylabel("CDF of volume of flows", fontsize=FONT_SIZE)
ax.set_title("Flow-volume CDF by workload", fontsize=FONT_SIZE)
ax.set_ylim(0, 1)
ax.tick_params(axis="both", labelsize=FONT_SIZE)
ax.legend(frameon=True, fontsize=FONT_SIZE)
fig.tight_layout()
plt.show()


In [ ]:
from matplotlib.lines import Line2D

OUTPUT_FILE = "sec6_flow_cdfs.png"
FONT_SIZE = 16
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

for name, df in distributions.items():
    ax1.step(
        df["flow_size"],
        df["flow_cdf"],
        where="post",
        linewidth=2,
        linestyle="-",
        color=colors[name],
    )
    ax2.step(
        df["flow_size"],
        df["volume_cdf"],
        where="post",
        linewidth=2,
        linestyle="--",
        color=colors[name],
    )

ax1.set_xscale("log")
ax1.set_xlabel("Flow size (bytes)", fontsize=FONT_SIZE)
ax1.set_ylabel("CDF of number of flows", fontsize=FONT_SIZE)
ax2.set_ylabel("CDF of volume of flows", fontsize=FONT_SIZE)
ax1.set_ylim(0, 1)
ax2.set_ylim(0, 1)
ax1.tick_params(axis="both", labelsize=FONT_SIZE)
ax2.tick_params(axis="y", labelsize=FONT_SIZE)
ax1.grid(True)
ax2.grid(False)

workload_handles = [
    Line2D([0], [0], color=colors[name], lw=2, label=name) for name in distributions
]
metric_handles = [
    Line2D([0], [0], color="black", lw=2, linestyle="-", label="Flow count"),
    Line2D([0], [0], color="black", lw=2, linestyle="--", label="Flow volume"),
]

LEGEND_NCOL = 3


def row_major(items, ncol):
    # matplotlib's legend() fills column-major; re-order the input so the
    # rendered grid reads row-major (1 2 3 / 4 5) instead of (1 3 5 / 2 4).
    return [item for col in range(ncol) for item in items[col::ncol]]


ax1.legend(
    handles=row_major(workload_handles + metric_handles, LEGEND_NCOL),
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncol=LEGEND_NCOL,
    frameon=True,
    fontsize=FONT_SIZE - 2,
    columnspacing=1.2,
    handletextpad=0.5,
)

fig.tight_layout()
fig.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")
plt.show()
